# 04 — Model Comparison: Decision Tree vs Random Forest


**Target:** `log_ClosePrice`

In [1]:
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

RANDOM_STATE = 42
TARGET = "log_ClosePrice"


## 1. Load data and baseline

In [2]:
train = pd.read_csv("sfr_train_model_ready.csv")
test = pd.read_csv("sfr_test_model_ready.csv")

X_train, y_train = train.drop(columns=[TARGET]), train[TARGET]
X_test, y_test = test.drop(columns=[TARGET]), test[TARGET]

baseline = pd.read_csv("model_results.csv")
BASELINE_R2 = baseline.loc[baseline.model == "LinearRegression", "r2_test"].iloc[0]
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Baseline (LinearRegression) test R2: {BASELINE_R2}")

Train: (49607, 29), Test: (11996, 29)
Baseline (LinearRegression) test R2: 0.8643


In [3]:
def evaluate(model, name):
    """Fit, score, and return a results row."""
    model.fit(X_train, y_train)
    pred_tr, pred_te = model.predict(X_train), model.predict(X_test)
    row = {
        "model": name,
        "notebook": "04_model_comparison",
        "r2_train": round(r2_score(y_train, pred_tr), 4),
        "r2_test": round(r2_score(y_test, pred_te), 4),
        "rmse_test_log": round(np.sqrt(mean_squared_error(y_test, pred_te)), 4),
        "mae_test_dollars": round(mean_absolute_error(np.exp(y_test), np.exp(pred_te)), 0),
    }
    print(f"{name}: R2 train={row['r2_train']:.4f}  test={row['r2_test']:.4f}  "
          f"MAE=${row['mae_test_dollars']:,.0f}")
    return row

## 2. Decision Tree


In [4]:
dt_full = evaluate(DecisionTreeRegressor(random_state=RANDOM_STATE), "DecisionTree (full depth)")

DecisionTree (full depth): R2 train=0.9999  test=0.8275  MAE=$288,149


In [5]:
depth_results = []
for depth in [4, 6, 8, 10, 12, 16]:
    m = DecisionTreeRegressor(max_depth=depth, random_state=RANDOM_STATE).fit(X_train, y_train)
    depth_results.append({
        "max_depth": depth,
        "r2_train": round(r2_score(y_train, m.predict(X_train)), 4),
        "r2_test": round(r2_score(y_test, m.predict(X_test)), 4),
    })
depth_df = pd.DataFrame(depth_results)
depth_df["gap"] = (depth_df.r2_train - depth_df.r2_test).round(4)
depth_df

,max_depth,r2_train,r2_test,gap
0,4,0.7874,0.7812,0.0062
1,6,0.8471,0.8395,0.0076
2,8,0.8810,0.8660,0.0150
3,10,0.9086,0.8703,0.0383
4,12,0.9327,0.8684,0.0643
5,16,0.9710,0.8566,0.1144


In [6]:
best_depth = int(depth_df.loc[depth_df.r2_test.idxmax(), "max_depth"])
dt_tuned = evaluate(DecisionTreeRegressor(max_depth=best_depth, random_state=RANDOM_STATE),
                    f"DecisionTree (max_depth={best_depth})")

DecisionTree (max_depth=10): R2 train=0.9086  test=0.8703  MAE=$255,082


## 3. Random Forest

In [7]:
rf = RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1)
rf_row = evaluate(rf, "RandomForest (200 trees)")

RandomForest (200 trees): R2 train=0.9887  test=0.9182  MAE=$193,550


## 4. Compare against baseline

In [8]:
results = pd.concat([baseline, pd.DataFrame([dt_full, dt_tuned, rf_row])], ignore_index=True)
results = results.drop_duplicates(subset="model", keep="last")
results["vs_baseline"] = (results.r2_test - BASELINE_R2).round(4)
results = results.sort_values("r2_test", ascending=False).reset_index(drop=True)
results.to_csv("model_results.csv", index=False)
results

,model,notebook,r2_train,r2_test,rmse_test_log,mae_test_dollars,vs_baseline
0,RandomForest (200 trees),04_model_comparison,0.9887,0.9182,0.1925,193550.0,0.0539
1,DecisionTree (max_depth=10),04_model_comparison,0.9086,0.8703,0.2424,255082.0,0.0060
2,LinearRegression,03_baseline_model,0.8659,0.8643,0.2480,262260.0,0.0000
3,DecisionTree (full depth),04_model_comparison,0.9999,0.8275,0.2796,288149.0,-0.0368


## 5. Random Forest Feature Importance

In [9]:
imp = pd.Series(rf.feature_importances_, index=X_train.columns).sort_values(ascending=False)
imp.head(10).round(4)

PostalCode_te        0.6037
LivingArea           0.1742
City_te              0.0921
Longitude            0.0209
CountyOrParish_te    0.0154
LotSizeSquareFeet    0.0150
Latitude             0.0143
YearBuilt            0.0114
DaysOnMarket         0.0088
MLSAreaMajor_te      0.0075
dtype: float64

## Recorded results & model behavior

| Model | R² test | vs baseline (0.8643) | R² train | MAE ($) |
|---|---|---|---|---|
| **Random Forest (200 trees)** | **0.9182** | **+0.0539** | 0.9887 | $193,550 |
| Decision Tree (max_depth=10) | 0.8703 | +0.0060 | 0.9086 | $255,082 |
| Decision Tree (full depth) | 0.8275 | -0.0368 | 0.9999 | $288,149 |
| Linear Regression (baseline) | 0.8643 | — | 0.8659 | $262,260 |

### Decision Tree — behavior

- **Strengths:** captures non-linearities and feature interactions the linear baseline cannot (e.g., location × size); needs no feature scaling; fully interpretable when shallow.
- **Weaknesses:** high variance. Grown to full depth it memorizes the training set (R² train = 0.9999) but generalizes worse on test — the classic overfitting signature. Even at its best depth (10), a single tree's test R² is limited because predictions are piecewise-constant averages over leaves; it can't extrapolate and is unstable to small data changes.

### Random Forest — behavior

- **Strengths:** averaging 200 bootstrapped, feature-subsampled trees cancels most of the single tree's variance while keeping the non-linearity — the test R² of 0.9182 is the best so far. Robust to outliers and irrelevant features; provides feature importances.
- **Weaknesses:** the train–test gap (R² 0.9887 → 0.9182) shows it still overfits noise somewhat; it cannot predict outside the range of training prices (no extrapolation); slower to train/predict and far less interpretable than the baseline; importances are spread across correlated location features, so they say "where" matters without saying how.
